# 🛡️ SOC L1 Log Analysis — Demo Notebook

This notebook walks through a complete Tier 1 analyst workflow:
1. Load and inspect raw logs
2. Parse each log type
3. Run detection rules
4. Visualize findings
5. Export an incident summary

---

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import sys
import os
import json
import re
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from collections import defaultdict
from datetime import datetime

# Add project root to path
sys.path.insert(0, os.path.abspath('..'))

print('✅ Imports successful')

## 1️⃣ Load Raw Log Files

In [ ]:
LOG_DIR = '../logs'

log_files = {
    'auth':    os.path.join(LOG_DIR, 'auth.log'),
    'syslog':  os.path.join(LOG_DIR, 'syslog.log'),
    'web':     os.path.join(LOG_DIR, 'apache_access.log'),
    'windows': os.path.join(LOG_DIR, 'windows_events.log'),
    'dns':     os.path.join(LOG_DIR, 'dns_queries.log'),
}

for name, path in log_files.items():
    exists = os.path.exists(path)
    lines  = sum(1 for _ in open(path)) if exists else 0
    status = '✅' if exists else '❌'
    print(f'{status}  {name:<10}  {lines:>4} lines   {path}')

## 2️⃣ Parse auth.log — Failed & Successful Logins

In [ ]:
FAILED_RE  = re.compile(r'(\w+ \d+ \d+:\d+:\d+).*Failed password for (?:invalid user )?(\S+) from (\d+\.\d+\.\d+\.\d+)')
SUCCESS_RE = re.compile(r'(\w+ \d+ \d+:\d+:\d+).*Accepted (?:password|publickey) for (\S+) from (\d+\.\d+\.\d+\.\d+)')

failed_rows  = []
success_rows = []

with open(log_files['auth']) as f:
    for line in f:
        m = FAILED_RE.search(line)
        if m:
            failed_rows.append({'timestamp': m.group(1), 'user': m.group(2), 'ip': m.group(3)})
            continue
        m = SUCCESS_RE.search(line)
        if m:
            success_rows.append({'timestamp': m.group(1), 'user': m.group(2), 'ip': m.group(3)})

df_failed  = pd.DataFrame(failed_rows)
df_success = pd.DataFrame(success_rows)

print(f'Failed logins  : {len(df_failed)}')
print(f'Successful logins: {len(df_success)}')
print()
print('── Sample Failed Logins ──')
df_failed.head(5)

## 3️⃣ Visualize Failed Logins by IP

In [ ]:
ip_counts = df_failed['ip'].value_counts().head(10)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(ip_counts.index[::-1], ip_counts.values[::-1], color='#e74c3c', edgecolor='#c0392b')

for bar, val in zip(bars, ip_counts.values[::-1]):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=10, color='#2c3e50')

ax.set_title('Top IPs by Failed SSH Login Attempts', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Number of Failed Attempts')
ax.set_ylabel('Source IP')
ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('../reports/failed_logins_by_ip.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to reports/failed_logins_by_ip.png')

## 4️⃣ Parse Windows Event Log

In [ ]:
df_win = pd.read_csv(log_files['windows'])
df_win['TimeCreated'] = pd.to_datetime(df_win['TimeCreated'])

print(f'Total events : {len(df_win)}')
print()
print('── Event ID Distribution ──')
print(df_win.groupby(['EventID','EventName']).size().reset_index(name='Count').to_string(index=False))

In [ ]:
event_counts = df_win['EventID'].value_counts()

EVENT_LABELS = {
    4624: '4624\nLogon Success',
    4625: '4625\nLogon Failure',
    4672: '4672\nPrivilege Use',
    4688: '4688\nProcess Create',
    4698: '4698\nTask Created',
    4732: '4732\nGroup Change',
    1102: '1102\nLog Cleared',
    4776: '4776\nCredential Val',
    4634: '4634\nLogoff',
}

colors = ['#2ecc71' if e in (4624,4634) else '#e74c3c' if e in (4625,4732,1102) else '#f39c12'
          for e in event_counts.index]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar([EVENT_LABELS.get(e, str(e)) for e in event_counts.index],
              event_counts.values, color=colors, edgecolor='white', linewidth=0.8)

for bar, val in zip(bars, event_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            str(val), ha='center', fontsize=10)

ax.set_title('Windows Security Event Distribution', fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel('Count')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('../reports/windows_event_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 5️⃣ Parse Apache Access Log — Attack Surface

In [ ]:
WEB_RE = re.compile(
    r'(?P<ip>\d+\.\d+\.\d+\.\d+) - \S+ \[(?P<ts>[^\]]+)\] '
    r'"(?P<method>\S+) (?P<uri>\S+) \S+" (?P<status>\d+) (?P<size>\d+) '
    r'"[^"]*" "(?P<ua>[^"]+)"'
)

web_rows = []
with open(log_files['web']) as f:
    for line in f:
        m = WEB_RE.match(line.strip())
        if m:
            web_rows.append(m.groupdict())

df_web = pd.DataFrame(web_rows)
df_web['status'] = df_web['status'].astype(int)

print(f'Total requests : {len(df_web)}')
print(f'Unique IPs     : {df_web["ip"].nunique()}')
print()
print('── Status Code Breakdown ──')
print(df_web['status'].value_counts().to_string())

In [ ]:
SCANNERS = ['sqlmap','nikto','nmap','nuclei','dirbuster','gobuster','masscan','acunetix']
df_web['is_scanner'] = df_web['ua'].str.lower().apply(
    lambda ua: any(s in ua for s in SCANNERS)
)

SQLI_PATTERNS = ["union select","or 1=1","drop table","sleep(","' or '","--"]
df_web['is_sqli'] = df_web['uri'].str.lower().apply(
    lambda u: any(p in u for p in SQLI_PATTERNS)
)

TRAVERSAL = ["../","etc/passwd","etc/shadow","..%2f"]
df_web['is_traversal'] = df_web['uri'].str.lower().apply(
    lambda u: any(p in u for p in TRAVERSAL)
)

threat_summary = {
    'Scanner Requests':     df_web['is_scanner'].sum(),
    'SQLi Attempts':        df_web['is_sqli'].sum(),
    'Path Traversal':       df_web['is_traversal'].sum(),
    '404 Responses':        (df_web['status'] == 404).sum(),
    '200 Responses':        (df_web['status'] == 200).sum(),
}

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#e74c3c','#c0392b','#e67e22','#95a5a6','#2ecc71']
bars = ax.bar(threat_summary.keys(), threat_summary.values(), color=colors, edgecolor='white')

for bar, val in zip(bars, threat_summary.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            str(val), ha='center', fontsize=11, fontweight='bold')

ax.set_title('Web Log Threat Breakdown', fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel('Request Count')
ax.spines[['top','right']].set_visible(False)
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig('../reports/web_threat_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()

## 6️⃣ DNS Beaconing Analysis

In [ ]:
DNS_RE = re.compile(
    r'(?P<ts>\S+) client (?P<client>\d+\.\d+\.\d+\.\d+)#\d+: query: (?P<domain>\S+) \S+ IN'
)

dns_rows = []
with open(log_files['dns']) as f:
    for line in f:
        m = DNS_RE.search(line)
        if m:
            d = m.groupdict()
            parts = d['domain'].rstrip('.').split('.')
            d['base_domain'] = '.'.join(parts[-2:]) if len(parts) >= 2 else d['domain']
            dns_rows.append(d)

df_dns = pd.DataFrame(dns_rows)
print(f'Total DNS queries : {len(df_dns)}')
print(f'Unique domains    : {df_dns["domain"].nunique()}')
print()

beaconing = df_dns.groupby(['client','base_domain']).size().reset_index(name='count')
beaconing = beaconing[beaconing['count'] >= 3].sort_values('count', ascending=False)
print('── Potential Beaconing (3+ queries to same domain) ──')
print(beaconing.to_string(index=False))

## 7️⃣ Combined Alert Summary

In [ ]:
alerts = []

# Brute force from auth log
for ip, count in df_failed['ip'].value_counts().items():
    if count >= 5:
        alerts.append({'severity':'HIGH',  'type':'Brute Force',       'source': ip,  'detail': f'{count} failed logins'})

# Success after failure
failed_ips  = set(df_failed['ip'])
success_ips = set(df_success['ip'])
for ip in failed_ips & success_ips:
    user = df_success[df_success['ip']==ip]['user'].iloc[0]
    alerts.append({'severity':'CRITICAL','type':'Success After BF',   'source': ip,  'detail': f'User: {user}'})

# Web scanners
for ip in df_web[df_web['is_scanner']]['ip'].unique():
    tool = df_web[(df_web['ip']==ip)&(df_web['is_scanner'])]['ua'].iloc[0][:30]
    alerts.append({'severity':'HIGH',  'type':'Web Scanner',          'source': ip,  'detail': tool})

# SQLi
for ip in df_web[df_web['is_sqli']]['ip'].unique():
    alerts.append({'severity':'CRITICAL','type':'SQL Injection',       'source': ip,  'detail': 'SQLi payload detected'})

# DNS beaconing
for _, row in beaconing.iterrows():
    alerts.append({'severity':'CRITICAL','type':'C2 Beaconing',        'source': row['client'], 'detail': f'{row["base_domain"]} x{row["count"]}'})

df_alerts = pd.DataFrame(alerts)
print(f'Total Alerts Generated: {len(df_alerts)}')
print()
print(df_alerts.to_string(index=False))

In [ ]:
sev_counts = df_alerts['severity'].value_counts()
colors_map = {'CRITICAL':'#e74c3c','HIGH':'#e67e22','MEDIUM':'#f1c40f','LOW':'#2ecc71'}
colors_pie = [colors_map.get(s,'#95a5a6') for s in sev_counts.index]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Pie — severity
ax1.pie(sev_counts.values, labels=sev_counts.index, colors=colors_pie,
        autopct='%1.0f%%', startangle=140,
        wedgeprops={'edgecolor':'white','linewidth':2})
ax1.set_title('Alert Severity Distribution', fontweight='bold')

# Bar — alert type
type_counts = df_alerts['type'].value_counts()
ax2.barh(type_counts.index[::-1], type_counts.values[::-1], color='#3498db', edgecolor='white')
for i, v in enumerate(type_counts.values[::-1]):
    ax2.text(v + 0.05, i, str(v), va='center', fontsize=10)
ax2.set_title('Alerts by Type', fontweight='bold')
ax2.set_xlabel('Count')
ax2.spines[['top','right']].set_visible(False)

plt.suptitle('SOC L1 — Daily Alert Summary', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../reports/alert_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved → reports/alert_summary.png')

## 8️⃣ Export Findings to JSON

In [ ]:
output = {
    'generated_at': datetime.utcnow().isoformat() + 'Z',
    'summary': {
        'total_failed_logins':    len(df_failed),
        'total_successful_logins':len(df_success),
        'total_web_requests':     len(df_web),
        'total_dns_queries':      len(df_dns),
        'total_alerts':           len(df_alerts),
    },
    'alerts': df_alerts.to_dict(orient='records')
}

with open('../reports/findings.json', 'w') as f:
    json.dump(output, f, indent=2)

print('✅ Findings exported to reports/findings.json')
print(json.dumps(output['summary'], indent=2))